# 🏒 Hockey Pythagoras Estimate
### Learning and Understanding the Hockey Pythagoras Estimate
*Data from MoneyPuck.com and Hockey-Reference, and Method from [Stapled To The Bench](https://www.stapledtothebench.com/wp-content/uploads/2024/12/2024-12-Hockey-Pythagoras-Estimate.pdf)*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.animation as animation
import seaborn as sns

print("Libraries loaded successfully!")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Libraries loaded successfully!


We will first need to clean and restructure our datasets, so they work with each other rather than against each other!

For the full database we are going to format the Season as 2008-09 to keep a common season format throught both databases, and other housekeeping items, such as deleting columns we won't need to use.

In [21]:
df_f = pd.read_csv("teams_2008_to_2024.csv")
df_f['season'] = df_f['season'].apply(lambda x: f"{x}-{str(x+1)[-2:]}")

df_f = df_f[df_f["situation"] == "all"]

df_f.loc[df_f['team'] == 'ARI', 'team'] = 'UTA'
df_f.loc[df_f['team'] == 'L.A', 'team'] = 'LAK'
df_f.loc[df_f['team'] == 'T.B', 'team'] = 'TBL'
df_f.loc[df_f['team'] == 'ATL', 'team'] = 'WPG'
df_f.loc[df_f['team'] == 'S.J', 'team'] = 'SJS'
df_f.loc[df_f['team'] == 'N.J', 'team'] = 'NJD'

df_f = df_f.drop(columns=['team.1', 'name', 'position', 'situation'])
df_f.head(5)


df_f['season'].describe()


count         522
unique         17
top       2024-25
freq           32
Name: season, dtype: object

Then the points database, we need to add the Arizona Coyotes scores onto UTA scores (name change) and then "flip" the database to match the formating of the final HPE dataset

In [22]:
df_pts = pd.read_csv("pts_uta.csv")
df_b = pd.read_csv("pts_ari.csv")


df_pts['UTA'] = df_pts['UTA'].combine_first(df_b['PHX'])


df_pts = df_pts[(df_pts["Season"] != '2025-26') & (df_pts["Season"] >= '2008-09')]
df_pts = df_pts.rename(columns={"Season":"season"})
df_pts = df_pts.drop(columns=['Rk','Lg'])

df_pts_long = df_pts.melt(id_vars='season', var_name='team', value_name='pts')
df_pts_long = df_pts_long.dropna()

df_pts_long['season'].describe()

count         522
unique         17
top       2024-25
freq           32
Name: season, dtype: object

Now, we will combine both datasets!

In [23]:
df_pts = df_f.merge(df_pts_long, on=['team', 'season'], how='left')
df_pts.head(5)


,team,season,games_played,xGoalsPercentage,corsiPercentage,fenwickPercentage,iceTime,xOnGoalFor,xGoalsFor,xReboundsFor,...,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,penalitiesFor,penalitiesAgainst,pts
0,MIN,2008-09,82,0.48,0.48,0.48,299195.0,2210.01,188.02,148.38,...,257.0,36.30,27.99,29.58,204.64,204.38,200.25,NaN,NaN,89.0
1,BOS,2008-09,81,0.51,0.50,0.50,295613.0,2386.17,223.59,160.11,...,293.0,34.80,35.63,36.91,212.87,210.92,206.34,NaN,NaN,116.0
2,UTA,2008-09,81,0.47,0.46,0.47,294089.0,2187.76,195.56,149.80,...,228.0,36.93,33.43,34.02,222.85,223.80,219.38,NaN,NaN,67.0
3,LAK,2008-09,82,0.51,0.51,0.51,300139.0,2448.00,225.29,174.04,...,471.0,36.07,29.23,30.51,222.34,222.74,217.17,NaN,NaN,79.0
4,COL,2008-09,82,0.49,0.47,0.48,299370.0,2207.19,203.61,156.31,...,291.0,34.83,36.92,37.93,205.01,206.99,203.10,NaN,NaN,69.0


In [24]:
def calculate_hpe(row):
    WHPE = round(82 * (row['goalsFor'] ** 2) / ( (row['goalsFor'] ** 2 ) + (row['goalsAgainst'] ** 2) ))
    
    THPE = round(WHPE * 0.22)
    
    LHPE = 82-WHPE-THPE
    
    return WHPE, LHPE, THPE

In [28]:
def calculate_pts(row):
    WHPE, LHPE, THPE = row['WLO']
    
    pts = (2 * WHPE) + THPE
    
    return pts    

In [30]:
df_HPE = df_pts[(df_pts["season"] != 2019) & (df_pts['season'] != 2013)].copy()
df_HPE['WLO'] = df_HPE.apply(calculate_hpe, axis=1)
df_HPE['p_pts'] = df_HPE.apply(calculate_pts, axis=1)


df_HPE[['season','team','WLO', 'p_pts', 'pts']].head(5)

,season,team,WLO,p_pts,pts
0,2008-09,MIN,"(44, 28, 10)",98,89.0
1,2008-09,BOS,"(55, 15, 12)",122,116.0
2,2008-09,UTA,"(32, 43, 7)",71,67.0
3,2008-09,LAK,"(36, 38, 8)",80,79.0
4,2008-09,COL,"(30, 45, 7)",67,69.0
